In [ ]:
%pip install -U weaviate-agents
%pip install sentence_transformers
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import weaviate
import weaviate.classes as wvc
from typing import List
import requests
import re

In [3]:
print(weaviate.__version__)

4.21.0


In [13]:
client = weaviate.connect_to_local()

print(client.is_ready())

True


In [14]:
def download_and_chunk(src_url: str, chunk_size: int, overlap_size: int) -> List[str]:
    response = requests.get(src_url)  # Retrieve source text
    source_text = re.sub(r"\s+", " ", response.text)  # Remove multiple whitespaces
    text_words = re.split(r"\s", source_text)  # Split text by single whitespace

    chunks = []
    for i in range(0, len(text_words), chunk_size):  # Iterate through & chunk data
        chunk = " ".join(text_words[max(i - overlap_size, 0): i + chunk_size])  # Join a set of words into a string
        chunks.append(chunk)
    return chunks


pro_git_chapter_url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"
chunked_text = download_and_chunk(pro_git_chapter_url, 150, 25)

In [16]:
collection_name = "GitBookChunk"

if client.collections.exists(collection_name):  # In case we've created this collection before
    client.collections.delete(collection_name)  # THIS WILL DELETE ALL DATA IN THE COLLECTION

chunks = client.collections.create(
    name=collection_name,
    properties=[
        wvc.config.Property(
            name="chunk",
            data_type=wvc.config.DataType.TEXT
        ),
        wvc.config.Property(
            name="chapter_title",
            data_type=wvc.config.DataType.TEXT
        ),
        wvc.config.Property(
            name="chunk_index",
            data_type=wvc.config.DataType.INT
        ),
    ],
    # vector_config=wvc.config.Configure.Vectors.none()
)

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\nithishkumar\Personal Data\Projects\Python projects\Data Science\Python Learn\ai_projects\intelliget_document_QandA_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4271.06it/s]


In [18]:
with chunks.batch.dynamic() as batch:
    for i, chunk in enumerate(chunked_text):

        vector = model.encode(chunk).tolist()

        batch.add_object(
            properties={
                "chapter_title": "What is Git",
                "chunk": chunk,
                "chunk_index": i
            },
            vector=vector
        )

c:\Users\nithishkumar\Personal Data\Projects\Python projects\Data Science\Python Learn\ai_projects\intelliget_document_QandA_bot\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\nithishkumar\AppData\Local\Temp\ipykernel_26112\793643893.py:1: ResourceWarning: unclosed <socket.socket fd=1480, family=23, type=1, proto=0, laddr=('::1', 63792, 0, 0), raddr=('::1', 8080, 0, 0)>
  with chunks.batch.dynamic() as batch:


In [19]:
query = "distributed version control"

query_vector = model.encode(query).tolist()

response = chunks.query.near_vector(
    near_vector=query_vector,
    limit=3
)

for obj in response.objects:
    print(obj.properties["chunk"])
    print("=" * 80)

either ask a remote server to do it or pull an older version of the file from the remote server to do it locally. This also means that there is very little you can't do if you're offline or off VPN. If you get on an airplane or a train and want to do a little work, you can commit happily (to your _local_ copy, remember?) until you get to a network connection to upload. If you go home and can't get your VPN client working properly, you can still work. In many other systems, doing so is either impossible or painful. In Perforce, for example, you can't do much when you aren't connected to the server; in Subversion and CVS, you can edit files, but you can't commit changes to your database (because your database is offline). This may not seem like a huge deal, but you may be surprised what a big difference it can make. ==== Git Has Integrity Everything in Git is checksummed before it is stored and is then referred
the Git directory and placed on disk for you to use or modify. The staging ar

In [32]:
SYSTEM_PROMPT = """
    You are an assistant for answering questions about Git, a distributed version control system. 
    You have access to a collection of text chunks from the Pro Git book, which you can use to find relevant information to answer user questions.
    When a user asks a question, you should:
    1. Search the collection for relevant chunks of text that can help answer the question.
    2. Combine the relevant information from the chunks to provide a comprehensive answer.
    3. only use information from the chunks to answer the question, and do not include any information that is not present in the chunks.
"""

In [ ]:
SYSTEM_DICT = {
            "role": "system",
            "content": SYSTEM_PROMPT
}
messages = []
def chat_with_model(messages):
    """
    Function to interact with the Ollama API for chat-based interactions.
    
    Parameters:
    - messages: A list of dictionaries representing the conversation history. Each dictionary should have a 'role' (e.g., 'user', 'assistant') and 'content' (the message text).
    
    Returns:
    - The response from the model as a string.
    """
    try:
        URL = "http://localhost:11434/api/chat"
        data = {
            "model": "gemma3:270m",
            "messages": [SYSTEM_DICT] + messages,
            "options": {
                "temperature": 0.7,
                "top_k": 50,
                "top_p": 0.5
            },
            "stream": False
        }

        response = requests.post(URL, json=data)
        res = dict(response.json())
        content = res['message']['content']
        return content
    except Exception as e:
        print(f"An error occurred: {e}")

In [58]:
def answer_question(prompt):

    query_vector = model.encode(prompt).tolist()

    response_chunks = chunks.query.near_vector(
        near_vector=query_vector,
        limit=1
    )
    final_prompt = prompt + " \n chunks:\n " + " ".join([chunk.properties["chunk"] for chunk in response_chunks.objects])
    print("prompt: ", final_prompt)
    messages.append({"role": "user", "content": final_prompt})
    response = chat_with_model(messages)
    print("response: ", response)
    messages.append({"role": "assistant", "content": response})
    # return response

In [61]:
answer_question("What mechanism git use for checksumming?")

prompt:  What mechanism git use for checksumming? 
 chunks:
 surprised what a big difference it can make. ==== Git Has Integrity Everything in Git is checksummed before it is stored and is then referred to by that checksum. This means it's impossible to change the contents of any file or directory without Git knowing about it. This functionality is built into Git at the lowest levels and is integral to its philosophy. You can't lose information in transit or get file corruption without Git being able to detect it. The mechanism that Git uses for this checksumming is called a SHA-1 hash.(((SHA-1))) This is a 40-character string composed of hexadecimal characters (0–9 and a–f) and calculated based on the contents of a file or directory structure in Git. A SHA-1 hash looks something like this: [source] ---- 24b9da6552252987aa493b52f8696cd6d3b00373 ---- You will see these hash values all over the place in Git because it uses them so much. In fact, Git stores everything in its database not 